# Notebook 03 - Explainable AI with Grad-CAM and Grad-CAM++

**Project:** Explainable Deep Learning for MRI Brain Tumor Classification and Segmentation Using Transfer Learning, MONAI, U-Net, and Grad-CAM

Run notebooks in order. Each notebook writes outputs into the same project folder so later notebooks can reuse them.


## Purpose

This notebook loads the best trained classifier and applies:

- Grad-CAM
- Grad-CAM++

The output is a set of heatmap overlays showing which MRI regions influenced the predicted tumor class.


In [ ]:
import sys, subprocess, json, random
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "torch", "torchvision", "grad-cam", "opencv-python",
                           "pandas", "numpy", "matplotlib", "pillow"])
print("Running in Colab:", IN_COLAB)


In [ ]:
from pathlib import Path
import json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torchvision import models, transforms

from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
if IN_COLAB:
    PROJECT_ROOT = Path("/content/brain_tumor_xai_project")
else:
    PROJECT_ROOT = Path.cwd() / "brain_tumor_xai_project"

config = json.loads((PROJECT_ROOT / "project_config.json").read_text())
PROJECT_ROOT = Path(config["project_root"])
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = Path(config["model_dir"])
FIGURE_DIR = Path(config["figure_dir"])
XAI_DIR = FIGURE_DIR / "xai_gradcam"; XAI_DIR.mkdir(parents=True, exist_ok=True)

metadata = pd.read_csv(config["classification_metadata_csv"])
class_names = json.loads(Path(config["class_names_json"]).read_text())
num_classes = len(class_names)
IMAGE_SIZE = int(config.get("image_size", 224))
print("Classes:", class_names)


In [ ]:
class BasicCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.4), nn.Linear(256, num_classes))
    def forward(self, x):
        return self.classifier(self.features(x))

def build_model(model_name, num_classes):
    if model_name == "basic_cnn":
        return BasicCNN(num_classes)
    if model_name == "resnet50":
        m = models.resnet50(weights=None); m.fc = nn.Linear(m.fc.in_features, num_classes); return m
    if model_name == "densenet121":
        m = models.densenet121(weights=None); m.classifier = nn.Linear(m.classifier.in_features, num_classes); return m
    if model_name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=None); m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes); return m
    raise ValueError(model_name)

def get_target_layer(model, model_name):
    if model_name == "basic_cnn": return model.features[-3]
    if model_name == "resnet50": return model.layer4[-1]
    if model_name == "densenet121": return model.features[-1]
    if model_name == "efficientnet_b0": return model.features[-1]
    raise ValueError(model_name)


In [ ]:
MODEL_NAME = "resnet50"  # change if your best model is different
ckpt = torch.load(MODEL_DIR / f"{MODEL_NAME}_best.pt", map_location=device)
model = build_model(ckpt["model_name"], len(ckpt["class_names"]))
model.load_state_dict(ckpt["state_dict"])
model = model.to(device).eval()
target_layer = get_target_layer(model, MODEL_NAME)
print("Loaded model:", MODEL_NAME)
print("Grad-CAM target layer:", target_layer)


In [ ]:
preprocess = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def load_image_for_cam(path):
    pil_img = Image.open(path).convert("RGB")
    rgb_resized = pil_img.resize((IMAGE_SIZE, IMAGE_SIZE))
    rgb_float = np.float32(rgb_resized) / 255.0
    input_tensor = preprocess(pil_img).unsqueeze(0).to(device)
    return rgb_float, input_tensor

@torch.no_grad()
def predict(input_tensor):
    logits = model(input_tensor)
    probs = torch.softmax(logits, dim=1)[0]
    pred_idx = int(torch.argmax(probs).item())
    confidence = float(probs[pred_idx].item())
    return pred_idx, confidence


In [ ]:
pred_path = OUTPUT_DIR / f"{MODEL_NAME}_test_predictions.csv"
pred_df = pd.read_csv(pred_path) if pred_path.exists() else metadata[metadata["split"]=="test"].copy()

N_EXAMPLES = 8
if "pred_class" in pred_df.columns:
    correct = pred_df[pred_df["class_name"] == pred_df["pred_class"]]
    wrong = pred_df[pred_df["class_name"] != pred_df["pred_class"]]
    selected = pd.concat([
        correct.sample(min(len(correct), max(1, N_EXAMPLES//2)), random_state=SEED),
        wrong.sample(min(len(wrong), max(0, N_EXAMPLES//2)), random_state=SEED) if len(wrong) else correct.sample(0)
    ]).head(N_EXAMPLES)
else:
    selected = pred_df.sample(min(N_EXAMPLES, len(pred_df)), random_state=SEED)
selected.head()


In [ ]:
cam = GradCAM(model=model, target_layers=[target_layer])
cam_pp = GradCAMPlusPlus(model=model, target_layers=[target_layer])

rows = []
for idx, row in selected.reset_index(drop=True).iterrows():
    rgb_float, input_tensor = load_image_for_cam(row["path"])
    pred_idx, confidence = predict(input_tensor)
    targets = [ClassifierOutputTarget(pred_idx)]

    heatmap = cam(input_tensor=input_tensor, targets=targets)[0]
    heatmap_pp = cam_pp(input_tensor=input_tensor, targets=targets)[0]
    overlay = show_cam_on_image(rgb_float, heatmap, use_rgb=True)
    overlay_pp = show_cam_on_image(rgb_float, heatmap_pp, use_rgb=True)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(rgb_float); axes[0].set_title(f"Original\nTrue: {row['class_name']}")
    axes[1].imshow(overlay); axes[1].set_title(f"Grad-CAM\nPred: {class_names[pred_idx]} ({confidence:.2f})")
    axes[2].imshow(overlay_pp); axes[2].set_title("Grad-CAM++")
    for ax in axes: ax.axis("off")
    plt.tight_layout()
    out_path = XAI_DIR / f"{MODEL_NAME}_xai_example_{idx+1}.png"
    plt.savefig(out_path, dpi=200)
    plt.show()

    np.save(XAI_DIR / f"{MODEL_NAME}_example_{idx+1}_gradcam.npy", heatmap)
    rows.append({
        "image_path": row["path"], "true_class": row["class_name"],
        "pred_class": class_names[pred_idx], "confidence": confidence,
        "correct": row["class_name"] == class_names[pred_idx],
        "gradcam_image": str(out_path)
    })

xai_df = pd.DataFrame(rows)
xai_df.to_csv(OUTPUT_DIR / f"{MODEL_NAME}_xai_gradcam_summary.csv", index=False)
xai_df


In [ ]:
def heatmap_mask_overlap(heatmap, mask, heatmap_threshold=0.6):
    '''
    Optional function for comparing a Grad-CAM heatmap with a tumor mask.
    heatmap: 2D numpy array with values 0-1
    mask: 2D binary tumor mask with same shape
    '''
    heatmap_bin = heatmap >= heatmap_threshold
    mask_bin = mask > 0
    intersection = np.logical_and(heatmap_bin, mask_bin).sum()
    union = np.logical_or(heatmap_bin, mask_bin).sum()
    return np.nan if union == 0 else intersection / union


## Outputs from Notebook 03

- Grad-CAM and Grad-CAM++ overlay PNG images
- raw Grad-CAM `.npy` heatmaps
- `*_xai_gradcam_summary.csv`

Next: run `04_monai_segmentation.ipynb`.
